# Unified Perceptual Autoencoder (All Categories)

This notebook consolidates the best-performing anomaly detection architectural choices across the team into a single, unified pipeline.

### Core Features:
1. **Multi-Category Support**: Automates the pipeline across all 15 MVTec AD categories, rather than hardcoding.
2. **Convolutional Bottleneck (V6b)**: Preserves spatial arrangements rather than flattening, which is critical for perceptual loss.
3. **ResNet-18 Perceptual Loss**: Uses a frozen PatchCore backbone (ResNet-18) to penalize differences in deep feature space, forcing the decoder to generate sharp edges and realistic textures.
4. **SSIM + L1 Loss**: Augments the perceptual loss with pixel-level Structural Similarity and absolute error to stabilize color and luminance.
5. **Otsu + Canny Masking**: Employs dynamic object-centered masking to prevent the model from memorizing normal background pixels.
6. **Explicit Seeding**: Ensures deterministic training to prevent the random initialization collapse discovered in earlier phases.\n

## 1. Imports and Setup
We import standard PyTorch utilities along with `torchmetrics` for the SSIM calculation. The notebook is configured to run on a CUDA-enabled GPU if available.

If you are running this in **Google Colab**, the cell below will automatically install required dependencies (`torchmetrics`) and download the MVTec AD dataset for you!\n

In [1]:
import os
import sys

# Google Colab Setup
if 'google.colab' in sys.modules:
    print("Detected Google Colab. Installing dependencies and downloading dataset...")
    !pip install -q torchmetrics
    
    # Create data directory
    os.makedirs("../../data/raw/mvtec_ad", exist_ok=True)
    
    # Download and extract dataset if it doesn't exist
    if not os.path.exists("../../data/raw/mvtec_ad/screw"):
        print("Downloading MVTec AD dataset (~5GB)... This may take a few minutes.")
        !wget -q https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz
        !tar -xf mvtec_anomaly_detection.tar.xz -C ../../data/raw/mvtec_ad
        !rm mvtec_anomaly_detection.tar.xz
        print("Dataset downloaded and extracted!")

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from pathlib import Path
import numpy as np
from PIL import Image
from sklearn.metrics import average_precision_score, roc_auc_score
from torchmetrics.functional.image import structural_similarity_index_measure as ssim

print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path("../../data/raw/mvtec_ad")
SIZE = 256
BATCH_SIZE = 16
EPOCHS = 40

PyTorch Version: 2.5.1+cu121
GPU Available: True


## 2. Dynamic Data Loading and Masking
Unlike previous notebooks hardcoded for a single category, this loader accepts any MVTec AD category dynamically.

**Otsu+Canny Object Masking:**
To prevent the autoencoder from memorizing the background (which artificially deflates the loss and ruins anomaly detection), we apply an object-centered mask using a combination of Otsu thresholding and Canny edge detection during the `__getitem__` pipeline.\n

In [2]:
def load_category_paths(category):
    cat_dir = DATA_ROOT / category
    if not cat_dir.exists():
        raise FileNotFoundError(f"Category {category} not found at {cat_dir}")
        
    train_good = sorted((cat_dir / "train" / "good").glob("*.png"))
    test_good = sorted((cat_dir / "test" / "good").glob("*.png"))

    test_defect, test_defect_masks = [], []
    test_dir = cat_dir / "test"
    if test_dir.exists():
        for defect_dir in sorted(test_dir.iterdir()):
            if not defect_dir.is_dir() or defect_dir.name == "good":
                continue
            for img_path in sorted(defect_dir.glob("*.png")):
                mask_path = cat_dir / "ground_truth" / defect_dir.name / f"{img_path.stem}_mask.png"
                test_defect.append(img_path)
                test_defect_masks.append(mask_path)

    return train_good, test_good, test_defect, test_defect_masks

def extract_otsu_canny_mask(img_np):
    """Extract object mask using Otsu thresholding + Canny edges to isolate the object from background."""
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Otsu
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Canny
    edges = cv2.Canny(blurred, 50, 150)
    
    # Combine
    combined = cv2.bitwise_or(thresh, edges)
    
    # Morphological closing to fill holes
    kernel = np.ones((5,5), np.uint8)
    closed = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel)
    
    # Normalize to 0-1
    return (closed > 0).astype(np.float32)

class MVTecDataset(Dataset):
    def __init__(self, image_paths, is_train=True):
        self.image_paths = image_paths
        self.is_train = is_train
        self.transform = T.Compose([
            T.Resize((SIZE, SIZE)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")
        
        # We need the numpy array for Otsu+Canny masking
        img_resized_np = np.array(img.resize((SIZE, SIZE)))
        mask = extract_otsu_canny_mask(img_resized_np)
        
        x = self.transform(img)
        return x, torch.tensor(mask).unsqueeze(0)

## 3. Convolutional Autoencoder Architecture
We use an Encoder-Bottleneck-Decoder architecture. Crucially, the bottleneck is **convolutional** rather than flattened into a linear layer. This ensures that the spatial relationships of the features are preserved when the reconstructed image is passed up to the ResNet-18 critic for perceptual loss evaluation.\n

In [3]:
class ConvAutoencoder(nn.Module):
    def __init__(self, bottleneck_channels=32):
        super().__init__()
        # Encoder (using ELU for smoother gradients)
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ELU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ELU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ELU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(256), nn.ELU(),
            # Bottleneck
            nn.Conv2d(256, bottleneck_channels, kernel_size=3, stride=1, padding=1)
        )
        
        # Decoder
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(bottleneck_channels, 256, kernel_size=3, stride=1, padding=1), nn.BatchNorm2d(256), nn.ELU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(128), nn.ELU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ELU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ELU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid() # Scale to [0,1]
        )

    def forward(self, x):
        z = self.enc(x)
        return self.dec(z)

## 4. Tri-Fold Loss (Perceptual + SSIM + L1)
Here we define the core innovation of this notebook:
1. **ResNet-18 Perceptual Loss**: We load an ImageNet-pretrained ResNet-18 and freeze it. We pass both the original and reconstructed images through it and calculate the MSE of their feature maps at Layers 1 and 2. This heavily penalizes the autoencoder if it produces blurry textures lacking sharp edges.
2. **SSIM**: We use `torchmetrics` to evaluate overarching structural layout.
3. **L1 (MAE)**: We calculate the absolute error of pixels to guarantee color and brightness stability.

`Total Loss = 0.84 * (1 - SSIM) + 0.16 * L1 + 0.05 * Perceptual_MSE`\n

In [4]:
class PerceptualLossModule(nn.Module):
    def __init__(self):
        super().__init__()
        # Load frozen ResNet-18 (Same backbone as PatchCore)
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        resnet = models.resnet18(weights=weights).eval()
        
        for param in resnet.parameters():
            param.requires_grad = False
            
        # We extract features from the early/mid layers which capture textures and edges
        self.layer1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1)
        self.layer2 = nn.Sequential(self.layer1, resnet.layer2)
        
        # ImageNet normalization required for ResNet
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
    def forward(self, y_true, y_pred):
        # Normalize inputs for ResNet
        y_true_norm = self.normalize(y_true)
        y_pred_norm = self.normalize(y_pred)
        
        # Extract features
        true_f1 = self.layer1(y_true_norm)
        pred_f1 = self.layer1(y_pred_norm)
        
        true_f2 = self.layer2(y_true_norm)
        pred_f2 = self.layer2(y_pred_norm)
        
        # MSE between feature maps
        loss_l1 = F.mse_loss(pred_f1, true_f1)
        loss_l2 = F.mse_loss(pred_f2, true_f2)
        
        # Average the perceptual loss across layers
        return (loss_l1 + loss_l2) / 2.0

def combined_loss_fn(y_pred, y_true, perceptual_module, alpha=0.84, beta=0.16, gamma=0.05):
    """
    Loss = 0.84 * (1-SSIM) + 0.16 * L1 + 0.05 * Perceptual_MSE
    """
    # 1. SSIM
    # We use default settings (11x11 gaussian filter)
    ssim_val = ssim(y_pred, y_true, data_range=1.0)
    loss_ssim = 1.0 - ssim_val
    
    # 2. L1 (MAE)
    loss_l1 = F.l1_loss(y_pred, y_true)
    
    # 3. Perceptual
    loss_perceptual = perceptual_module(y_true, y_pred)
    
    total_loss = alpha * loss_ssim + beta * loss_l1 + gamma * loss_perceptual
    return total_loss

## 5. Training Loop
The training loop wraps everything up and applies **Global Seeding** (`torch.manual_seed(42)`) before each category loop. This prevents the "seed collapse" problem where random weight initialization causes the autoencoder to produce completely black images.

Finally, during training, we multiply our reconstructed images by the Otsu+Canny mask so that the loss function completely ignores the background pixels!\n

In [ ]:
def train_and_evaluate_category(category):
    print(f"{'='*40}")
    print(f"Training Category: {category}")
    print(f"{'='*40}")
    
    # 1. Determinism
    torch.manual_seed(42)
    np.random.seed(42)
    
    # 2. Data
    try:
        train_g, test_g, test_d, test_m = load_category_paths(category)
    except FileNotFoundError as e:
        print(e)
        return
        
    train_ds = MVTecDataset(train_g, is_train=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    
    # 3. Model
    model = ConvAutoencoder().to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    perceptual_module = PerceptualLossModule().to(DEVICE)
    
    # 4. Training
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for x, mask in train_loader:
            x, mask = x.to(DEVICE), mask.to(DEVICE)
            
            optimizer.zero_grad()
            recon = model(x)
            
            # Mask the background so loss is only calculated on the object
            x_masked = x * mask
            recon_masked = recon * mask
            
            loss = combined_loss_fn(recon_masked, x_masked, perceptual_module)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * x.size(0)
            
        epoch_loss /= len(train_ds)
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f}")
            
    print(f"Finished training {category}.")
    return model

SyntaxError: unterminated f-string literal (detected at line 2) (2970816961.py, line 2)

## 6. Evaluation and Visualization
After training, we pass defective images through the autoencoder. The model should struggle to reconstruct the defects (since it was only trained on normal images). We can pinpoint anomalies by calculating the pixel-wise difference between the original image and the reconstruction.

This block visualizes:
1. The Original Defective Image
2. The Autoencoder's Reconstruction
3. The Anomaly Heatmap (Pixel-wise Error)
4. The Ground Truth Mask\n

In [ ]:
import matplotlib.pyplot as plt

def evaluate_and_visualize_category(model, category, num_images=3):
    model.eval()
    
    try:
        _, _, test_defect, test_defect_masks = load_category_paths(category)
    except FileNotFoundError:
        return
        
    if len(test_defect) == 0:
        print(f"No defect images found for {category}.")
        return
        
    # We'll just visualize a few images
    test_ds = MVTecDataset(test_defect[:num_images], is_train=False)
    
    fig, axes = plt.subplots(num_images, 4, figsize=(16, 4 * num_images))
    if num_images == 1:
        axes = [axes]
        
    for idx in range(num_images):
        x, _ = test_ds[idx]
        x_input = x.unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            recon = model(x_input)
            
        # Convert tensors back to numpy for matplotlib
        original_img = x.permute(1, 2, 0).cpu().numpy()
        recon_img = recon.squeeze(0).permute(1, 2, 0).cpu().numpy()
        
        # Anomaly map: Mean Absolute Error per pixel (average across color channels)
        anomaly_map = np.mean(np.abs(original_img - recon_img), axis=-1)
        
        # Load ground truth mask
        gt_mask_path = test_defect_masks[idx]
        if gt_mask_path.exists():
            gt_mask = np.array(Image.open(gt_mask_path).resize((SIZE, SIZE))) > 0
        else:
            gt_mask = np.zeros((SIZE, SIZE))
            
        # Plotting
        ax = axes[idx]
        
        ax[0].imshow(original_img)
        ax[0].set_title("Original Image")
        ax[0].axis('off')
        
        ax[1].imshow(recon_img)
        ax[1].set_title("Reconstruction")
        ax[1].axis('off')
        
        im = ax[2].imshow(anomaly_map, cmap='jet')
        ax[2].set_title("Anomaly Heatmap")
        ax[2].axis('off')
        fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)
        
        ax[3].imshow(gt_mask, cmap='gray')
        ax[3].set_title("Ground Truth Mask")
        ax[3].axis('off')
        
    plt.tight_layout()
    plt.show()

In [ ]:
# We can run this for 'screw' first as a test, but it supports iterating over all categories
test_categories = ["screw"] # change to os.listdir(DATA_ROOT) to run all
for cat in test_categories:
    trained_model = train_and_evaluate_category(cat)
    evaluate_and_visualize_category(trained_model, cat)